## **1. Data Load & Overview**

In [1]:
# 1.1 Load Data
import pandas as pd
import re
import nltk
import contractions
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

df = pd.read_csv("../1_data/processed/01_cleaned.csv")
print("Data loaded successfully!")
print(f"Shape: {df.shape}")

Data loaded successfully!
Shape: (22980, 16)


In [2]:
# 1.2 Download NLTK Data
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [3]:
# 1.3 Check Review Columns
for review in df["Review_Title"].values[:5]:
    print(review)
    print("---")

"pretty decent airline"
---
"Not a good airline"
---
"flight was fortunately short"
---
"I will never fly again with Adria"
---
"it ruined our last days of holidays"
---


In [4]:
for review in df["Review"].values[:5]:
    print(review)
    print("---")

  Moroni to Moheli. Turned out to be a pretty decent airline. Online booking worked well, checkin and boarding was fine and the plane looked well maintained. Its a very short flight - just 20 minutes or so so i didn't expect much but they still managed to hand our a bottle of water and some biscuits which i though was very nice. Both flights on time.
---
 Moroni to Anjouan. It is a very small airline. My ticket advised me to turn up at 0800hrs which I did. There was confusion at this small airport. I was then directed to the office of AB Aviation which was still closed. It opened at 0900hrs and I was told that the flight had been put back to 1300hrs and that they had tried to contact me. This could not be true as they did not have my phone number. I was with a local guide and he had not been informed either. I presume that I was bumped off. The later flight did operate but as usual, there was confusion at check-in. The flight was only 30mins and there were no further problems. Not a go

> ### **Text Input Strategy - Full Text (Title + Review)**
>
> The review title is prepended to the review body before preprocessing. As observed in the sample data, review titles tend to capture the core sentiment in a concise form (e.g., *"I will never fly again with Adria"*) and do not simply repeat the review body. Combining both is expected to enrich the sentiment signal, particularly for shorter reviews where the title may provide critical context.

In [5]:
# 1.4 Concatinate "Review_Title" to "Review"
df['Full_Review'] = df['Review_Title'].str.strip('"') + '. ' + df['Review'].str.strip()

# 1.5 Drop original columns
df = df.drop(columns=['Review_Title', 'Review'])

for review in df["Full_Review"].values[:5]:
    print(review)
    print("---")

pretty decent airline. Moroni to Moheli. Turned out to be a pretty decent airline. Online booking worked well, checkin and boarding was fine and the plane looked well maintained. Its a very short flight - just 20 minutes or so so i didn't expect much but they still managed to hand our a bottle of water and some biscuits which i though was very nice. Both flights on time.
---
Not a good airline. Moroni to Anjouan. It is a very small airline. My ticket advised me to turn up at 0800hrs which I did. There was confusion at this small airport. I was then directed to the office of AB Aviation which was still closed. It opened at 0900hrs and I was told that the flight had been put back to 1300hrs and that they had tried to contact me. This could not be true as they did not have my phone number. I was with a local guide and he had not been informed either. I presume that I was bumped off. The later flight did operate but as usual, there was confusion at check-in. The flight was only 30mins and 

## **2. Text Pre-processing**


> ### **Preprocessing Strategy by Feature Set**
>
> - **Set A (Numerical Only)**: This set relies on numerical sub-ratings as features. Hence, text pre-processing is not required.
>
> - **Set B (VADER)**: Minimal preprocessing to preserve sentiment signals such as negations, punctuation emphasis (!, ?), and sentence structure. Stopword removal and tokenization are intentionally excluded as they may distort VADER's sentiment scoring.
>
> - **Set C (Rule-based Aspect Sentiment)**: Full preprocessing pipeline applied to improve keyword matching accuracy. Contraction expansion is performed before stopword removal to ensure negation words (e.g., not) are retained as separate tokens.
>
> - **Set D (VADER + Rule-based)**: Combines the outputs of Set B and Set C. No additional preprocessing required as each component follows its respective pipeline.
>
> - **Set E (ABSA BERT)**: Minimal preprocessing similar to Set B. Stopword removal and lemmatization are excluded as BERT relies on full sentence context and handles subword tokenization (WordPiece) internally.

| Technique | Set B | Set C | Set E |
|---|---|---|---|
| Lowercasing | ✅ | ✅ | ✅ |
| HTML / Special Character Removal | ✅ | ✅ | ✅ |
| Whitespace Normalization | ✅ | ✅ | ✅ |
| Contraction Expansion | ✅ | ✅ (before stopword removal) | ✅ |
| Punctuation Removal (partial) | ✅ | ✅ | ✅ |
| Stopword Removal | ❌ | ✅ | ❌ |
| Tokenization | ❌ | ✅ | ❌ |
| Lemmatization | ❌ | ✅ | ❌ |

In [6]:
# 2.1 Preprocessing Function (Set B and E)

def preprocess_vader_bert(text):
    if pd.isna(text):
        return ""
    # Contraction Expansion
    text = contractions.fix(text)
    # Lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    # Remove URLs
    text = re.sub(r'http\S+', ' ', text)
    # Remove special characters (keep !, ?, ., ,)
    text = re.sub(r'[^a-zA-Z0-9\s!?.,]', ' ', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [7]:
# 2.2 Preprocessing Function (Set C)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_rule_based(text):
    if pd.isna(text):
        return ""
    # Contraction Expansion
    text = contractions.fix(text)
    # Lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    # Remove URLs
    text = re.sub(r'http\S+', ' ', text)
    # Remove all punctuation and special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]
    # Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    # Remove single character tokens
    tokens = [t for t in tokens if len(t) > 1]
    return ' '.join(tokens)

In [8]:
# 2.3 Apply Preprocessing (Set B and E)
print("Applying Set B/E preprocessing...")
df['cleaned_review_BE'] = df['Full_Review'].apply(preprocess_vader_bert)
print("Done!")

Applying Set B/E preprocessing...
Done!


In [9]:
# 2.4 Apply Preprocessing (Set C)
print("Applying Set C preprocessing...")
df['cleaned_review_C'] = df['Full_Review'].apply(preprocess_rule_based)
print("Done!")

Applying Set C preprocessing...
Done!


## **3. Verify Results**

In [10]:
df.head(3)

,Airline Name,Verified,Type Of Traveller,Seat Type,Seat Comfort,Cabin Staff Service,Food & Beverages,Ground Service,Inflight Entertainment,Wifi & Connectivity,Value For Money,Recommended,Covid_Period,review_length,Full_Review,cleaned_review_BE,cleaned_review_C
0,AB Aviation,True,Solo Leisure,Economy Class,4.0,5.0,4.0,4.0,NaN,NaN,3.0,1,0,352,pretty decent airline. Moroni to Moheli. Turne...,pretty decent airline. moroni to moheli. turne...,pretty decent airline moroni moheli turned pre...
1,AB Aviation,True,Solo Leisure,Economy Class,2.0,2.0,1.0,1.0,NaN,NaN,2.0,0,0,689,Not a good airline. Moroni to Anjouan. It is a...,not a good airline. moroni to anjouan. it is a...,good airline moroni anjouan small airline tick...
2,AB Aviation,True,Solo Leisure,Economy Class,2.0,1.0,1.0,1.0,NaN,NaN,2.0,0,0,405,flight was fortunately short. Anjouan to Dzaou...,flight was fortunately short. anjouan to dzaou...,flight fortunately short anjouan dzaoudzi smal...


In [11]:
for i, review in enumerate(df[['Full_Review', 'cleaned_review_BE', 'cleaned_review_C']].values[:50]):
    print(f"=== Review {i+1} ===")
    print(f"Original:\n{review[0]}\n")
    print(f"BE:\n{review[1]}\n")
    print(f"C:\n{review[2]}\n")

=== Review 1 ===
Original:
pretty decent airline. Moroni to Moheli. Turned out to be a pretty decent airline. Online booking worked well, checkin and boarding was fine and the plane looked well maintained. Its a very short flight - just 20 minutes or so so i didn't expect much but they still managed to hand our a bottle of water and some biscuits which i though was very nice. Both flights on time.

BE:
pretty decent airline. moroni to moheli. turned out to be a pretty decent airline. online booking worked well, checkin and boarding was fine and the plane looked well maintained. its a very short flight just 20 minutes or so so i did not expect much but they still managed to hand our a bottle of water and some biscuits which i though was very nice. both flights on time.

C:
pretty decent airline moroni moheli turned pretty decent airline online booking worked well checkin boarding fine plane looked well maintained short flight 20 minute expect much still managed hand bottle water biscuit

> ### **Scraping Error Removal**
>
> During web scraping, the Skytrax review pages included a recurring template phrase at the beginning of certain reviews in the format:
> *"[Airline Name] customer review. [review content]..."*
>
> For example (Starting from Review #43):
> - *"Adria Airways customer review. Flew Zurich-Ljubljana..."*
> - *"Afriqiyah Airways customer review. Gatwick to..."*
>
> This phrase carries no sentiment or aspect information and is therefore removed prior to preprocessing.

In [12]:
# Check how many reviews contain "customer review" pattern
pattern_count = df['Full_Review'].str.contains('customer review', case=False, na=False).sum()
print(f"Reviews containing 'customer review': {pattern_count}")

# See examples from different airlines
mask = df['Full_Review'].str.contains('customer review', case=False, na=False)
df[mask][['Airline Name', 'Full_Review']].head(10)

Reviews containing 'customer review': 4032


,Airline Name,Full_Review
42,Adria Airways,Adria Airways customer review. Outbound flight...
43,Adria Airways,Adria Airways customer review. Flew Zurich-Lju...
44,Adria Airways,Adria Airways customer review. Adria serves th...
45,Adria Airways,Adria Airways customer review. WAW-SKJ Economy...
46,Adria Airways,Adria Airways customer review. Sarajevo-Frankf...
47,Adria Airways,Adria Airways customer review. I had flights f...
48,Adria Airways,Adria Airways customer review. LJU to FRA and ...
49,Adria Airways,Adria Airways customer review. On my Ljubljana...
50,Adria Airways,Adria Airways customer review. Flights from LJ...
51,Adria Airways,Adria Airways customer review. I was very sati...


In [13]:
# Remove "[Airline Name] customer review" boilerplate
def remove_customer_review(text):
    if pd.isna(text):
        return text
    # Remove pattern like "Airline Name customer review."
    text = re.sub(r'[\w\s]+ customer review\.?\s*', '', text, flags=re.IGNORECASE)
    return text.strip()

# Apply to Full_Review before preprocessing
df['Full_Review'] = df['Full_Review'].apply(remove_customer_review)

In [14]:
# Re-apply preprocessing after boilerplate removal

print("Re-applying Set B/E preprocessing...")
df['cleaned_review_BE'] = df['Full_Review'].apply(preprocess_vader_bert)
print("Done!")

print("Re-applying Set C preprocessing...")
df['cleaned_review_C'] = df['Full_Review'].apply(preprocess_rule_based)
print("Done!")

Re-applying Set B/E preprocessing...
Done!
Re-applying Set C preprocessing...
Done!


In [15]:
for i, review in enumerate(df[['Full_Review', 'cleaned_review_BE', 'cleaned_review_C']].values[:50]):
    print(f"=== Review {i+1} ===")
    print(f"Original:\n{review[0]}\n")
    print(f"BE:\n{review[1]}\n")
    print(f"C:\n{review[2]}\n")

=== Review 1 ===
Original:
pretty decent airline. Moroni to Moheli. Turned out to be a pretty decent airline. Online booking worked well, checkin and boarding was fine and the plane looked well maintained. Its a very short flight - just 20 minutes or so so i didn't expect much but they still managed to hand our a bottle of water and some biscuits which i though was very nice. Both flights on time.

BE:
pretty decent airline. moroni to moheli. turned out to be a pretty decent airline. online booking worked well, checkin and boarding was fine and the plane looked well maintained. its a very short flight just 20 minutes or so so i did not expect much but they still managed to hand our a bottle of water and some biscuits which i though was very nice. both flights on time.

C:
pretty decent airline moroni moheli turned pretty decent airline online booking worked well checkin boarding fine plane looked well maintained short flight 20 minute expect much still managed hand bottle water biscuit

## **4. Save Processed Dataset**

In [17]:
import os
os.makedirs('../1_data/processed', exist_ok=True)
out_path = '../1_data/processed/02_preprocessed.csv'
df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print("\n")
print(f"Final shape: {df.shape}")
df.info()

Saved → ../1_data/processed/02_preprocessed.csv


Final shape: (22980, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22980 entries, 0 to 22979
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Airline Name            22980 non-null  object 
 1   Verified                22980 non-null  bool   
 2   Type Of Traveller       22980 non-null  object 
 3   Seat Type               22980 non-null  object 
 4   Seat Comfort            18763 non-null  float64
 5   Cabin Staff Service     18673 non-null  float64
 6   Food & Beverages        14162 non-null  float64
 7   Ground Service          18292 non-null  float64
 8   Inflight Entertainment  10245 non-null  float64
 9   Wifi & Connectivity     5896 non-null   float64
 10  Value For Money         21806 non-null  float64
 11  Recommended             22980 non-null  int64  
 12  Covid_Period            22980 non-null  int64  
 13  review_length   